# XLKD-02: Dense On-the-Fly Mapped Logit-KD Loss Preflight

Notebook kiểm tra khả năng **tối ưu hóa thực tế** của nhánh ablation:

> ExpansionNet v2 → S1 MobileNetV3-Large, Dense On-the-Fly
> Cross-Vocabulary Mapped Logit KD.

XLKD-01B đã khóa protocol sau khi top-32/64/128 không giữ đủ khối lượng xác
suất. XLKD-02 không thay ngưỡng cũ và không dùng sparse approximation.

## Phạm vi

- bốn ảnh Train+Restval cố định theo SHA256 seed 42;
- S1 khởi tạo từ `s1_mobilenetv3_seed42_best.pt`;
- Teacher frozen, `T=4`, `alpha_human=alpha_kd=0,5`;
- kiểm tra mapped mass, loss, backward, gradient ratio, overfit 20 bước và
  peak GPU;
- không đọc Validation, không mở Test, không authorize full training.

## Làm rõ mask mà không đổi phương pháp

CE dùng mask của **human target**. KL dùng mask của **pseudo-caption đã ánh
xạ**, vì Teacher và Student phải được teacher-force trên cùng chuỗi từ và có
cùng số vị trí. Hai kênh được chuẩn hóa riêng rồi mới phối hợp 0,5/0,5.

## Kaggle inputs bắt buộc

1. XLKD-00 output.
2. XLKD-01B output.
3. Phase 01 Formal Output.
4. Phase 02A Shared Student Pipeline.
5. Phase 02B S1 seed-42 baseline output.
6. Phase 03B ExpansionNet v2 preflight output.
7. Phase 03E ExpansionNet v2 handoff output.
8. ExpansionNet v2 offline model package.
9. MS COCO 2014 image dataset.

Runtime: GPU T4 hoặc mạnh hơn, Internet Off.


## Cell 1 — Authorization, integrity boundary và environment


In [1]:
import csv, gc, hashlib, json, math, os, pickle, platform, random, sys, time
from argparse import Namespace
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch

STARTED = time.perf_counter()
SEED = 42
INPUT_ROOT = Path(os.environ.get("KLCN_INPUT_ROOT", "/kaggle/input"))
NOTEBOOK_ROOT = (
    INPUT_ROOT / "notebooks"
    if (INPUT_ROOT / "notebooks").is_dir()
    else INPUT_ROOT
)
DATASET_ROOT = (
    INPUT_ROOT / "datasets"
    if (INPUT_ROOT / "datasets").is_dir()
    else INPUT_ROOT
)
OUT = Path(os.environ.get(
    "KLCN_OUTPUT_ROOT",
    "/kaggle/working//xlkd02_dense_mapped_loss_preflight",
))
AUDIT = OUT / "audits"
CONFIG = OUT / "configs"
RESULT = OUT / "results"
MANIFEST = OUT / "manifests"
for directory in [OUT, AUDIT, CONFIG, RESULT, MANIFEST]:
    directory.mkdir(parents=True, exist_ok=True)

def now():
    return datetime.now(timezone.utc).isoformat()

def read_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)

def write_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def exactly_one(name):
    # Kaggle may mount many notebook outputs. Build one filename index and
    # reuse it instead of recursively scanning once per required artifact.
    matches = sorted(_DISCOVERY_INDEX.get(name, []))
    assert len(matches) == 1, (
        f"Cần đúng một {name} dưới {NOTEBOOK_ROOT}; "
        f"tìm thấy {len(matches)}: {matches}"
    )
    return matches[0].resolve()

def resolve_inside(root, relative_path):
    root = Path(root).resolve()
    target = (root / relative_path).resolve()
    assert target.is_relative_to(root), (root, relative_path, target)
    return target

def verify_manifest(marker_path, marker):
    reference = marker["manifest"]["csv"]
    path = resolve_inside(marker_path.parent, reference["relative_path"])
    assert path.is_file() and sha256(path) == reference["sha256"]
    with path.open("r", encoding="utf-8", newline="") as handle:
        rows = list(csv.DictReader(handle))
    assert len(rows) == int(reference["rows"])
    for row in rows:
        artifact = resolve_inside(marker_path.parent, row["relative_path"])
        assert artifact.is_file()
        assert artifact.stat().st_size == int(row["size_bytes"])
        assert sha256(artifact) == row["sha256"]
    return rows

def set_deterministic(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.use_deterministic_algorithms(True, warn_only=False)
    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cuda.enable_mem_efficient_sdp(False)
    torch.backends.cuda.enable_math_sdp(True)

def find_expansionnet_manifest():
    matches = []
    preferred = INPUT_ROOT / "models"
    search_root = preferred if preferred.is_dir() else INPUT_ROOT
    for candidate in search_root.rglob("MODEL_SOURCE_MANIFEST.json"):
        try:
            payload = read_json(candidate)
        except Exception:
            continue
        if payload.get("model_id") == "expansionnet_v2":
            matches.append(candidate.resolve())
    assert len(matches) == 1, (
        "Cần đúng một ExpansionNet v2 offline Model input; "
        f"tìm thấy {len(matches)}: {matches}"
    )
    return matches[0]

_DISCOVERY_INDEX = {}
for _candidate in NOTEBOOK_ROOT.rglob("*"):
    if _candidate.is_file():
        _DISCOVERY_INDEX.setdefault(_candidate.name, []).append(
            _candidate.resolve()
        )

if not torch.cuda.is_available():
    raise RuntimeError(
        "XLKD-02 yêu cầu GPU. Chọn GPU T4 hoặc mạnh hơn rồi chạy lại."
    )
set_deterministic(SEED)
device = torch.device("cuda")

X01B_PATH = exactly_one("XLKD01B_DENSE_ONTHEFLY_REENTRY_COMPLETE.json")
AUTH_PATH = exactly_one(
    "XLKD02_DENSE_MAPPED_LOGIT_LOSS_PREFLIGHT_AUTHORIZED.json"
)
X00_PATH = exactly_one("XLKD00_EXPANSIONNET_S1_COMPATIBILITY_COMPLETE.json")
X00_AUTH_PATH = exactly_one(
    "XLKD01_GPU_PROBABILITY_MASS_PREFLIGHT_AUTHORIZED.json"
)
P1_PATH = exactly_one("PHASE1_FORMAL_DELIVERABLES_READY.json")
P2A_PATH = exactly_one("PHASE2_SHARED_TRAINING_BUNDLE_READY.json")
S1_PATH = exactly_one("S1_SEED42_FULL_RUN_COMPLETE.json")
P3B_PATH = exactly_one(
    "PHASE3_TEACHER_A_EXPANSIONNET_V2_PREFLIGHT_READY.json"
)
P3E_PATH = exactly_one(
    "PHASE3_TEACHER_A_EXPANSIONNET_V2_HANDOFF_COMPLETE.json"
)
MODEL_MANIFEST_PATH = find_expansionnet_manifest()
MODEL_ROOT = MODEL_MANIFEST_PATH.parent

X01B, AUTH, X00, X00_AUTH, P1, P2A, S1, P3B, P3E = map(
    read_json,
    [X01B_PATH, AUTH_PATH, X00_PATH, X00_AUTH_PATH,
     P1_PATH, P2A_PATH, S1_PATH,
     P3B_PATH, P3E_PATH],
)
assert X01B["status"] == "complete"
assert X01B["sparse_branch_status"] == "stopped_and_retained"
assert X01B["adaptive_branch"] == "dense_on_the_fly"
assert X01B["mapped_mass_gate"] is True
assert X01B["protocol_locked"] is True
assert X01B["xlkd02_authorized"] is True
assert X01B["full_training_authorized"] is False
assert AUTH["status"] == "authorized"
assert AUTH["physical_batch_size"] == 1
assert AUTH["fixed_image_count"] == 4
assert float(AUTH["temperature"]) == 4.0
assert float(AUTH["alpha_kd"]) == 0.5
assert AUTH["full_training_authorized"] is False
assert X00["status"] == "complete"
assert X00["structural_mapping_gate"] == "passed"
assert X00_AUTH["status"] == "authorized"
assert P1["status"] == "ready"
assert P2A["status"] == "ready"
assert S1["status"] == "complete"
assert S1["student_id"] == "S1" and int(S1["seed"]) == 42
assert P3B["status"] == "ready" and P3B["teacher"] == "ExpansionNet v2"
assert P3E["status"] == "complete"
for marker in [X01B, AUTH, X00, X00_AUTH, S1, P3B, P3E]:
    assert marker["test_set_status"] == "sealed"
    assert marker["test_set_used"] is False
assert P1["test_set_status"] == "sealed" and P1["test_used"] is False
assert P2A["data_policy"]["test_set_status"] == "sealed"
assert P2A["data_policy"]["test_set_used"] is False

x01b_rows = verify_manifest(X01B_PATH, X01B)
protocol_ref = AUTH["protocol"]
PROTOCOL_PATH = resolve_inside(
    X01B_PATH.parent, protocol_ref["relative_path"]
)
assert sha256(PROTOCOL_PATH) == protocol_ref["sha256"]
PROTOCOL = read_json(PROTOCOL_PATH)
assert PROTOCOL["status"] == "locked"
assert PROTOCOL["branch_type"] == (
    "adaptive_after_preregistered_sparse_topk_failure"
)
assert PROTOCOL["method_name"] == (
    "Dense On-the-Fly Cross-Vocabulary Mapped Logit KD"
)
assert PROTOCOL["full_training_authorized"] is False
GATE = PROTOCOL["xlkd02_fixed_batch"]["success_gate"]

write_json(AUDIT / "xlkd02_cell01_entry.json", {
    "schema_version": "xlkd02_cell01_v1",
    "created_at_utc": now(),
    "status": "passed",
    "xlkd01b_manifest_rows": len(x01b_rows),
    "sparse_branch_status": "stopped_and_retained",
    "adaptive_branch": "dense_on_the_fly",
    "protocol_locked": True,
    "gpu": torch.cuda.get_device_name(0),
    "python": platform.python_version(),
    "pytorch": torch.__version__,
    "strict_deterministic": True,
    "full_training_authorized": False,
    "validation_read": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})
print("=" * 78)
print("XLKD-02 CELL 1 — AUTHORIZATION")
print("=" * 78)
print("Status                    : PASSED")
print("Sparse branch             : STOPPED AND RETAINED")
print("Adaptive branch           : DENSE ON-THE-FLY")
print("XLKD-01B manifest         :", len(x01b_rows), "/", len(x01b_rows))
print("GPU                       :", torch.cuda.get_device_name(0))
print("Full training             : NOT AUTHORIZED")
print("Validation / Test         : NOT READ / SEALED")
print("Progress                  : 1 / 6 code cells")


XLKD-02 CELL 1 — AUTHORIZATION
Status                    : PASSED
Sparse branch             : STOPPED AND RETAINED
Adaptive branch           : DENSE ON-THE-FLY
XLKD-01B manifest         : 7 / 7
GPU                       : Tesla T4
Full training             : NOT AUTHORIZED
Validation / Test         : NOT READ / SEALED
Progress                  : 1 / 6 code cells


## Cell 2 — Vocabulary map, fixed four-image batch và dual preprocessing


In [2]:
STARTED = time.perf_counter()
import pyarrow.parquet as pq
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.transforms import InterpolationMode

mapping_ref = X00_AUTH["mapping_contract"]
MAPPING_PATH = resolve_inside(X00_PATH.parent, mapping_ref["relative_path"])
assert sha256(MAPPING_PATH) == mapping_ref["sha256"]
MAPPING_PAYLOAD = read_json(MAPPING_PATH)
mapping = {
    int(source): int(target)
    for source, target in MAPPING_PAYLOAD["mapping"].items()
}
assert len(mapping) == 9959
assert MAPPING_PAYLOAD["mapping_collisions"] == 0
assert float(MAPPING_PAYLOAD["vocabulary_count_coverage"]) == 0.9959

model_manifest = read_json(MODEL_MANIFEST_PATH)
assert model_manifest["model_id"] == "expansionnet_v2"

def verify_model_record(record):
    target = (MODEL_ROOT / record["relative_path"]).resolve()
    assert target.is_relative_to(MODEL_ROOT.resolve())
    assert target.is_file()
    assert target.stat().st_size == int(record["size_bytes"])
    assert sha256(target) == record["sha256"]
    return target

CHECKPOINT_PATH = verify_model_record(model_manifest["checkpoint"])
NATIVE_VOCAB_PATH = verify_model_record(model_manifest["vocabulary"])
for record in model_manifest["required_files"]:
    verify_model_record(record)
for record in model_manifest["integration_patches"]:
    target = (MODEL_ROOT / record["file"]).resolve()
    assert target.is_file() and sha256(target) == record["sha256"]

with NATIVE_VOCAB_PATH.open("rb") as handle:
    native_tokens = pickle.load(handle)
NATIVE_WORD2IDX = native_tokens["word2idx_dict"]
NATIVE_IDX2WORD = native_tokens["idx2word_list"]
NATIVE_BOS = int(NATIVE_WORD2IDX[native_tokens["sos_str"]])
NATIVE_EOS = int(NATIVE_WORD2IDX[native_tokens["eos_str"]])
NATIVE_PAD = int(NATIVE_WORD2IDX["PAD"])
NATIVE_UNK = int(NATIVE_WORD2IDX["UNK"])
assert (NATIVE_PAD, NATIVE_BOS, NATIVE_EOS, NATIVE_UNK) == (
    78, 79, 77, 80
)

PSEUDO_PATH = (
    P3E_PATH.parent / "pseudocaptions"
    / "teacher_a_expansionnet_v2_train_restval.jsonl"
).resolve()
PSEUDO_SUMMARY_PATH = (
    P3E_PATH.parent / "pseudocaptions"
    / "teacher_a_expansionnet_v2_pseudocaption_summary.json"
).resolve()
assert PSEUDO_PATH.is_file() and PSEUDO_SUMMARY_PATH.is_file()
PSEUDO_SUMMARY = read_json(PSEUDO_SUMMARY_PATH)
assert PSEUDO_SUMMARY["records"] == 113287
assert sha256(PSEUDO_PATH) == PSEUDO_SUMMARY["jsonl_sha256"]
assert PSEUDO_SUMMARY["held_out_test_overlap"] == 0
assert PSEUDO_SUMMARY["validation_overlap"] == 0

records = []
with PSEUDO_PATH.open("r", encoding="utf-8") as handle:
    for line in handle:
        row = json.loads(line)
        image_id = int(row["image_id"])
        records.append({
            "score": hashlib.sha256(
                f"{SEED}|{image_id}".encode("utf-8")
            ).hexdigest(),
            "image_id": image_id,
            "teacher_token_ids": [
                int(value) for value in row["teacher_token_ids"]
            ],
            "student_token_ids": [
                int(value) for value in row["student_token_ids"]
            ],
        })
assert len(records) == 113287
records.sort(key=lambda row: (row["score"], row["image_id"]))
FIXED_RECORDS = records[:4]
del records
assert len({row["image_id"] for row in FIXED_RECORDS}) == 4
for row in FIXED_RECORDS:
    assert len(row["teacher_token_ids"]) == len(row["student_token_ids"])
    assert [mapping[value] for value in row["teacher_token_ids"]] == (
        row["student_token_ids"]
    )

fixed_ids = [row["image_id"] for row in FIXED_RECORDS]
TRAIN_PARQUET = exactly_one("mscoco_train.parquet")
caption_table = pq.read_table(
    TRAIN_PARQUET,
    columns=["annotation_id", "image_id", "token_ids"],
    filters=[("image_id", "in", fixed_ids)],
)
captions = {}
for row in sorted(
    caption_table.to_pylist(),
    key=lambda value: (int(value["image_id"]), int(value["annotation_id"])),
):
    image_id = int(row["image_id"])
    captions.setdefault(image_id, [
        int(value) for value in row["token_ids"]
    ])
assert set(captions) == set(fixed_ids)
for row in FIXED_RECORDS:
    row["human_token_ids"] = captions[row["image_id"]]
    assert row["human_token_ids"][0] == 1
    assert row["human_token_ids"][-1] == 2
    assert max(row["human_token_ids"]) < 10080

IMAGE_MANIFEST_PATH = exactly_one("mscoco_image_manifest.parquet")
manifest_table = pq.read_table(
    IMAGE_MANIFEST_PATH,
    columns=["image_id", "experiment_split", "source_partition", "image_path"],
    filters=[("image_id", "in", fixed_ids)],
)
sample_manifest = {
    int(row["image_id"]): row for row in manifest_table.to_pylist()
}
assert set(sample_manifest) == set(fixed_ids)
assert all(
    sample_manifest[image_id]["experiment_split"] == "train"
    for image_id in fixed_ids
)

def find_image_leaf(name, prefix):
    matches = []
    for path in DATASET_ROOT.rglob(name):
        if path.is_dir() and next(path.glob(f"{prefix}*.jpg"), None):
            matches.append(path.resolve())
    matches = sorted(set(matches))
    assert len(matches) == 1, matches
    return matches[0]

TRAIN_LEAF = find_image_leaf("train2014", "COCO_train2014_")
VAL_LEAF = find_image_leaf("val2014", "COCO_val2014_")
for record in FIXED_RECORDS:
    source = sample_manifest[record["image_id"]]
    leaf = TRAIN_LEAF if source["source_partition"] == "train2014" else VAL_LEAF
    record["image_path"] = (leaf / Path(source["image_path"]).name).resolve()
    assert record["image_path"].is_file()

TEACHER_PREPROCESS = transforms.Compose([
    transforms.Resize((384, 384), interpolation=InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])
STUDENT_PREPROCESS = transforms.Compose([
    transforms.Resize(256, interpolation=InterpolationMode.BICUBIC),
    transforms.CenterCrop((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

class FixedDataset(Dataset):
    def __len__(self):
        return len(FIXED_RECORDS)

    def __getitem__(self, index):
        record = FIXED_RECORDS[index]
        with Image.open(record["image_path"]) as source:
            image = source.convert("RGB")
            teacher_image = TEACHER_PREPROCESS(image)
            student_image = STUDENT_PREPROCESS(image)
        return {
            "image_id": record["image_id"],
            "teacher_image": teacher_image,
            "student_image": student_image,
            "teacher_ids": torch.tensor(
                record["teacher_token_ids"], dtype=torch.long
            ),
            "student_pseudo_ids": torch.tensor(
                record["student_token_ids"], dtype=torch.long
            ),
            "human_ids": torch.tensor(
                record["human_token_ids"], dtype=torch.long
            ),
        }

def fixed_collate(rows):
    assert len(rows) == 1
    row = rows[0]
    return {
        "image_id": int(row["image_id"]),
        "teacher_image": row["teacher_image"].unsqueeze(0),
        "student_image": row["student_image"].unsqueeze(0),
        "teacher_ids": row["teacher_ids"].unsqueeze(0),
        "student_pseudo_ids": row["student_pseudo_ids"].unsqueeze(0),
        "human_ids": row["human_ids"].unsqueeze(0),
    }

FIXED_DATASET = FixedDataset()
FIXED_LOADER = DataLoader(
    FIXED_DATASET,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    collate_fn=fixed_collate,
)
probe = FIXED_DATASET[0]
assert probe["teacher_image"].shape == (3, 384, 384)
assert probe["student_image"].shape == (3, 224, 224)
assert torch.isfinite(probe["teacher_image"]).all()
assert torch.isfinite(probe["student_image"]).all()

SAMPLE_PATH = RESULT / "xlkd02_fixed_four_image_batch.json"
write_json(SAMPLE_PATH, {
    "schema_version": "xlkd02_fixed_batch_v1",
    "created_at_utc": now(),
    "seed": 42,
    "rule": "sort SHA256('42|image_id'), take first four",
    "split": "Train+Restval represented as experiment_split=train",
    "physical_batch_size": 1,
    "image_ids": fixed_ids,
    "human_caption_rule": "lowest annotation_id per image",
    "teacher_conditioning": "native pseudo-caption token IDs",
    "student_kd_conditioning": "aligned Phase-1 pseudo-caption token IDs",
    "ce_mask": "human target labels",
    "kd_mask": "aligned pseudo-caption target labels",
    "validation_read": False,
    "test_overlap": 0,
})
write_json(AUDIT / "xlkd02_cell02_fixed_batch.json", {
    "schema_version": "xlkd02_cell02_v1",
    "created_at_utc": now(),
    "status": "passed",
    "mapping_tokens": len(mapping),
    "mapping_collisions": 0,
    "fixed_images": 4,
    "sequence_alignment": "4 / 4",
    "teacher_input_shape": [3, 384, 384],
    "student_input_shape": [3, 224, 224],
    "mask_execution_clarification": {
        "human_ce": "human target labels",
        "mapped_kl": "aligned pseudo-caption target labels",
        "reason": "Teacher and Student KD positions must represent the same word sequence.",
    },
    "validation_read": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})
print("=" * 78)
print("XLKD-02 CELL 2 — FIXED BATCH")
print("=" * 78)
print("Vocabulary map            : 9,959 / 10,000, collisions 0")
print("Fixed images              : 4 / 4")
print("Teacher/Student alignment : 4 / 4")
print("Teacher input             : [3,384,384]")
print("Student input             : [3,224,224]")
print("Human CE / KD masks       : HUMAN / ALIGNED PSEUDO")
print("Validation / Test         : NOT READ / SEALED")
print("Progress                  : 2 / 6 code cells")


XLKD-02 CELL 2 — FIXED BATCH
Vocabulary map            : 9,959 / 10,000, collisions 0
Fixed images              : 4 / 4
Teacher/Student alignment : 4 / 4
Teacher input             : [3,384,384]
Student input             : [3,224,224]
Human CE / KD masks       : HUMAN / ALIGNED PSEUDO
Validation / Test         : NOT READ / SEALED
Progress                  : 2 / 6 code cells


## Cell 3 — Strict-load frozen Teacher và S1 baseline seed 42


In [3]:
STARTED = time.perf_counter()
import torch.nn as nn
import torch.nn.functional as F
from torch.torch_version import TorchVersion
from torchvision.models import mobilenet_v3_large

SOURCE_ROOT = MODEL_ROOT / "ExpansionNet_v2_src"
assert SOURCE_ROOT.is_dir()
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))
from models.End_ExpansionNet_v2 import End_ExpansionNet_v2

drop_args = Namespace(
    enc=0.0, dec=0.0, enc_input=0.0, dec_input=0.0, other=0.0
)
TEACHER = End_ExpansionNet_v2(
    swin_img_size=384, swin_patch_size=4, swin_in_chans=3,
    swin_embed_dim=192, swin_depths=[2, 2, 18, 2],
    swin_num_heads=[6, 12, 24, 48], swin_window_size=12,
    swin_mlp_ratio=4.0, swin_qkv_bias=True, swin_qk_scale=None,
    swin_drop_rate=0.0, swin_attn_drop_rate=0.0,
    swin_drop_path_rate=0.0, swin_norm_layer=torch.nn.LayerNorm,
    swin_ape=False, swin_patch_norm=True, swin_use_checkpoint=False,
    final_swin_dim=1536, d_model=512, N_enc=3, N_dec=3,
    num_heads=8, ff=2048, num_exp_enc_list=[32, 64, 128, 256, 512],
    num_exp_dec=16, output_word2idx=NATIVE_WORD2IDX,
    output_idx2word=NATIVE_IDX2WORD, max_seq_len=74,
    drop_args=drop_args, rank=device,
)
teacher_payload = torch.load(
    CHECKPOINT_PATH, map_location="cpu", weights_only=False
)
incompatible = TEACHER.load_state_dict(
    teacher_payload["model_state_dict"], strict=True
)
assert not incompatible.missing_keys and not incompatible.unexpected_keys
del teacher_payload
gc.collect()
TEACHER = TEACHER.to(device).eval()
for parameter in TEACHER.parameters():
    parameter.requires_grad_(False)
assert all(not parameter.requires_grad for parameter in TEACHER.parameters())

DECODER_CONTRACT = exactly_one("shared_transformer_decoder_contract.json")
decoder_config = read_json(DECODER_CONTRACT)["architecture"]
assert int(decoder_config["vocabulary_size"]) == 10080

class S1Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        model = mobilenet_v3_large(weights=None)
        self.backbone = model.features
        self.projection = nn.Linear(960, 256)
        self.visual_position_embedding = nn.Parameter(
            torch.empty(1, 49, 256)
        )
        self.output_norm = nn.LayerNorm(256)

    def forward(self, images):
        feature = self.backbone(images)
        tokens = feature.flatten(2).transpose(1, 2).contiguous()
        assert tokens.shape[1:] == (49, 960)
        return self.output_norm(
            self.projection(tokens) + self.visual_position_embedding
        )

class SharedDecoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.vocab = int(config["vocabulary_size"])
        self.d_model = int(config["d_model"])
        self.maximum_steps = int(config["maximum_decoder_steps"])
        self.token_embedding = nn.Embedding(
            self.vocab, self.d_model, padding_idx=0
        )
        self.position_embedding = nn.Embedding(
            self.maximum_steps, self.d_model
        )
        self.embedding_dropout = nn.Dropout(float(config["dropout"]))
        layer = nn.TransformerDecoderLayer(
            d_model=self.d_model,
            nhead=int(config["nhead"]),
            dim_feedforward=int(config["dim_feedforward"]),
            dropout=float(config["dropout"]),
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.transformer_decoder = nn.TransformerDecoder(
            layer, num_layers=int(config["num_layers"])
        )
        self.final_norm = nn.LayerNorm(self.d_model)
        self.output_bias = nn.Parameter(torch.zeros(self.vocab))

    def forward(self, ids, memory, mask):
        length = ids.shape[1]
        positions = torch.arange(
            length, device=ids.device
        ).unsqueeze(0).expand(ids.shape[0], -1)
        target = (
            self.token_embedding(ids) * math.sqrt(self.d_model)
            + self.position_embedding(positions)
        )
        target = self.embedding_dropout(target)
        causal = torch.triu(
            torch.ones(
                length, length, dtype=torch.bool, device=ids.device
            ), diagonal=1
        )
        hidden = self.transformer_decoder(
            target, memory, tgt_mask=causal,
            tgt_key_padding_mask=~mask.bool(),
        )
        hidden = self.final_norm(hidden)
        return F.linear(
            hidden, self.token_embedding.weight, self.output_bias
        )

STUDENT_ENCODER = S1Encoder()
STUDENT_DECODER = SharedDecoder(decoder_config)
BASELINE_CHECKPOINT = exactly_one("s1_mobilenetv3_seed42_best.pt")
assert sha256(BASELINE_CHECKPOINT) == S1["best_checkpoint"]["sha256"]
baseline_payload = torch.load(
    BASELINE_CHECKPOINT, map_location="cpu", weights_only=False
)
assert baseline_payload["schema_version"] == (
    "phase2_s1_full_training_checkpoint_v1"
)
STUDENT_ENCODER.load_state_dict(
    baseline_payload["encoder_state_dict"], strict=True
)
STUDENT_DECODER.load_state_dict(
    baseline_payload["decoder_state_dict"], strict=True
)
BASE_ENCODER_STATE = {
    key: value.detach().cpu().clone()
    for key, value in STUDENT_ENCODER.state_dict().items()
}
BASE_DECODER_STATE = {
    key: value.detach().cpu().clone()
    for key, value in STUDENT_DECODER.state_dict().items()
}
del baseline_payload
STUDENT_ENCODER = STUDENT_ENCODER.to(device)
STUDENT_DECODER = STUDENT_DECODER.to(device)

assert sum(p.numel() for p in STUDENT_ENCODER.parameters()) == 3231024
assert sum(p.numel() for p in STUDENT_DECODER.parameters()) == 5759328
assert sum(p.numel() for p in TEACHER.parameters()) > 100_000_000

mapped_src = torch.tensor(
    sorted(mapping), dtype=torch.long, device=device
)
mapped_dst = torch.tensor(
    [mapping[index] for index in sorted(mapping)],
    dtype=torch.long, device=device,
)
assert mapped_src.numel() == mapped_dst.numel() == 9959

def set_student_train():
    STUDENT_ENCODER.train()
    STUDENT_DECODER.train()
    count = 0
    for module in STUDENT_ENCODER.modules():
        if isinstance(module, nn.modules.batchnorm._BatchNorm):
            module.eval()
            count += 1
    assert count == 46

def reset_student_baseline():
    STUDENT_ENCODER.load_state_dict(BASE_ENCODER_STATE, strict=True)
    STUDENT_DECODER.load_state_dict(BASE_DECODER_STATE, strict=True)
    for parameter in list(STUDENT_ENCODER.parameters()) + list(STUDENT_DECODER.parameters()):
        parameter.grad = None

def teacher_mapped_distribution(batch, temperature=4.0):
    images = batch["teacher_image"].to(device, non_blocking=True)
    full = batch["teacher_ids"].to(device, non_blocking=True)
    decoder_input = full[:, :-1]
    labels = full[:, 1:]
    num_pads = (decoder_input == NATIVE_PAD).sum(dim=1).tolist()
    # Do not use inference_mode here: q is consumed by F.kl_div while
    # autograd records the Student branch. A tensor created in
    # inference_mode cannot be saved for that backward pass. no_grad keeps
    # the Teacher frozen but returns ordinary tensors that are valid targets.
    with torch.no_grad():
        logits = TEACHER(
            enc_x=images,
            dec_x=decoder_input,
            enc_x_num_pads=[0] * images.shape[0],
            dec_x_num_pads=[int(value) for value in num_pads],
            apply_log_softmax=False,
            mode="forward",
        )
        assert logits.shape == (
            images.shape[0], decoder_input.shape[1], 10000
        )
        probabilities = torch.softmax(logits.float() / temperature, dim=-1)
        selected = probabilities.index_select(-1, mapped_src)
        dense = torch.zeros(
            *selected.shape[:-1], 10080,
            dtype=selected.dtype, device=device,
        )
        index = mapped_dst.view(1, 1, -1).expand_as(selected)
        # XLKD-00 proved that the map is collision-free, so scatter (not
        # scatter_add) is mathematically equivalent and deterministic.
        dense.scatter_(-1, index, selected)
        mass = dense.sum(dim=-1)
        q = dense / mass.unsqueeze(-1)
        mask = labels.ne(NATIVE_PAD)
    return q.detach(), mask.detach(), mass.detach()

write_json(AUDIT / "xlkd02_cell03_models.json", {
    "schema_version": "xlkd02_cell03_v1",
    "created_at_utc": now(),
    "status": "passed",
    "teacher": "ExpansionNet v2",
    "teacher_strict_load": True,
    "teacher_frozen": True,
    "student": "S1 MobileNetV3-Large",
    "student_initialization": "seed-42 baseline best checkpoint",
    "student_checkpoint_sha256": sha256(BASELINE_CHECKPOINT),
    "student_parameters": 8990352,
    "mapped_teacher_dimensions": 9959,
    "student_vocabulary_size": 10080,
    "optimizer_created": False,
    "full_training_started": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})
print("=" * 78)
print("XLKD-02 CELL 3 — MODELS")
print("=" * 78)
print("Teacher strict-load       : PASSED / FROZEN")
print("Student initialization    : S1 SEED-42 BASELINE BEST")
print("Student parameters        : 8,990,352")
print("Mapped dimensions         : 9,959 -> 10,080")
print("Optimizer / full training : False / False")
print("Test Set status           : SEALED")
print("Progress                  : 3 / 6 code cells")


XLKD-02 CELL 3 — MODELS
Teacher strict-load       : PASSED / FROZEN
Student initialization    : S1 SEED-42 BASELINE BEST
Student parameters        : 8,990,352
Mapped dimensions         : 9,959 -> 10,080
Optimizer / full training : False / False
Test Set status           : SEALED
Progress                  : 3 / 6 code cells


## Cell 4 — Finite loss, backward, gradient ratio và GPU gate


In [4]:
STARTED = time.perf_counter()
TEMPERATURE = 4.0
ALPHA_HUMAN = 0.5
ALPHA_KD = 0.5

def to_device(batch):
    return {
        key: (
            value.to(device, non_blocking=True)
            if isinstance(value, torch.Tensor) else value
        )
        for key, value in batch.items()
    }

def student_losses(batch, q_teacher, kd_mask):
    current = to_device(batch)
    human = current["human_ids"]
    pseudo = current["student_pseudo_ids"]
    memory = STUDENT_ENCODER(current["student_image"])
    human_input, human_labels = human[:, :-1], human[:, 1:]
    pseudo_input, pseudo_labels = pseudo[:, :-1], pseudo[:, 1:]
    human_logits = STUDENT_DECODER(
        human_input, memory, human_input.ne(0)
    )
    pseudo_logits = STUDENT_DECODER(
        pseudo_input, memory, pseudo_input.ne(0)
    )
    human_ce = F.cross_entropy(
        human_logits.float().reshape(-1, 10080),
        human_labels.reshape(-1),
        ignore_index=0,
        reduction="mean",
    )
    assert pseudo_logits.shape[:2] == q_teacher.shape[:2]
    assert pseudo_logits.shape[-1] == q_teacher.shape[-1] == 10080
    student_log_prob = F.log_softmax(
        pseudo_logits.float() / TEMPERATURE, dim=-1
    )
    kl_positions = F.kl_div(
        student_log_prob, q_teacher, reduction="none"
    ).sum(dim=-1)
    valid = kd_mask.bool()
    assert torch.equal(valid, pseudo_labels.ne(0))
    kd_kl_t2 = (
        kl_positions[valid].mean() * (TEMPERATURE ** 2)
    )
    combined = ALPHA_HUMAN * human_ce + ALPHA_KD * kd_kl_t2
    return human_ce, kd_kl_t2, combined

reset_student_baseline()
set_student_train()
parameters = (
    list(STUDENT_ENCODER.parameters())
    + list(STUDENT_DECODER.parameters())
)
batch = next(iter(FIXED_LOADER))
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
q_teacher, kd_mask, mapped_mass = teacher_mapped_distribution(batch)
human_ce, kd_kl_t2, combined = student_losses(
    batch, q_teacher, kd_mask
)
weighted_ce = ALPHA_HUMAN * human_ce
weighted_kd = ALPHA_KD * kd_kl_t2

ce_grads = torch.autograd.grad(
    weighted_ce, parameters, retain_graph=True, allow_unused=True
)
kd_grads = torch.autograd.grad(
    weighted_kd, parameters, retain_graph=True, allow_unused=True
)
ce_grad_norm = math.sqrt(sum(
    float(grad.detach().float().pow(2).sum())
    for grad in ce_grads if grad is not None
))
kd_grad_norm = math.sqrt(sum(
    float(grad.detach().float().pow(2).sum())
    for grad in kd_grads if grad is not None
))
gradient_ratio = kd_grad_norm / max(ce_grad_norm, 1e-12)
for parameter in parameters:
    parameter.grad = None
combined.backward()
combined_grad_norm = float(torch.nn.utils.clip_grad_norm_(parameters, 1.0))
teacher_has_grad = any(
    parameter.grad is not None for parameter in TEACHER.parameters()
)
peak_memory_mib = torch.cuda.max_memory_allocated() / (1024 ** 2)

finite_loss_gate = all(math.isfinite(value) for value in [
    float(human_ce), float(kd_kl_t2), float(combined),
    ce_grad_norm, kd_grad_norm, combined_grad_norm,
])
mapped_mass_mean = float(mapped_mass[kd_mask].mean())
mapped_mass_minimum = float(mapped_mass[kd_mask].min())
mapped_mass_gate = (
    mapped_mass_mean >= float(GATE["mapped_mass_mean_min"])
    and mapped_mass_minimum >= float(GATE["mapped_mass_minimum_min"])
)
gradient_ratio_gate = (
    float(GATE["gradient_ratio_min"]) <= gradient_ratio
    <= float(GATE["gradient_ratio_max"])
)
teacher_gradient_gate = not teacher_has_grad
memory_gate = peak_memory_mib <= float(GATE["peak_gpu_memory_mib_max"])

LOSS_AUDIT = {
    "schema_version": "xlkd02_loss_backward_audit_v1",
    "created_at_utc": now(),
    "status": "passed",
    "image_id": batch["image_id"],
    "temperature": TEMPERATURE,
    "alpha_human_ce": ALPHA_HUMAN,
    "alpha_kd": ALPHA_KD,
    "human_ce": float(human_ce),
    "mapped_kl_t2": float(kd_kl_t2),
    "combined_loss": float(combined),
    "weighted_ce_gradient_norm": ce_grad_norm,
    "weighted_kd_gradient_norm": kd_grad_norm,
    "weighted_kd_to_ce_gradient_ratio": gradient_ratio,
    "combined_gradient_norm_before_clip": combined_grad_norm,
    "mapped_mass_mean": mapped_mass_mean,
    "mapped_mass_minimum": mapped_mass_minimum,
    "teacher_has_gradient": teacher_has_grad,
    "peak_gpu_memory_mib": peak_memory_mib,
    "gates": {
        "finite_loss_and_gradients": finite_loss_gate,
        "mapped_mass": mapped_mass_gate,
        "gradient_ratio": gradient_ratio_gate,
        "teacher_no_gradient": teacher_gradient_gate,
        "peak_gpu_memory": memory_gate,
    },
    "optimizer_step_performed": False,
    "full_training_started": False,
    "validation_read": False,
    "test_set_status": "sealed",
    "test_set_used": False,
}
write_json(RESULT / "xlkd02_loss_backward_gradient_ratio.json", LOSS_AUDIT)
write_json(AUDIT / "xlkd02_cell04_loss_backward.json", {
    **LOSS_AUDIT,
    "elapsed_seconds": time.perf_counter() - STARTED,
})
for parameter in parameters:
    parameter.grad = None
print("=" * 78)
print("XLKD-02 CELL 4 — LOSS / BACKWARD")
print("=" * 78)
print("Human CE                 :", round(float(human_ce), 6))
print("Mapped KL*T^2            :", round(float(kd_kl_t2), 6))
print("Combined loss            :", round(float(combined), 6))
print("Mapped mass mean / min   :", mapped_mass_mean, "/", mapped_mass_minimum)
print("KD/CE gradient ratio     :", round(gradient_ratio, 6))
print("Teacher gradient         :", teacher_has_grad)
print("Peak GPU memory          :", round(peak_memory_mib, 2), "MiB")
print("Finite / mass / ratio    :", finite_loss_gate, mapped_mass_gate, gradient_ratio_gate)
print("Teacher-grad / memory    :", teacher_gradient_gate, memory_gate)
print("Progress                 : 4 / 6 code cells")


XLKD-02 CELL 4 — LOSS / BACKWARD
Human CE                 : 2.231941
Mapped KL*T^2            : 17.264704
Combined loss            : 9.748322
Mapped mass mean / min   : 0.9963003993034363 / 0.9951349496841431
KD/CE gradient ratio     : 0.494502
Teacher gradient         : False
Peak GPU memory          : 1116.04 MiB
Finite / mass / ratio    : True True True
Teacher-grad / memory    : True True
Progress                 : 4 / 6 code cells


/tmp/ipykernel_23/3745410347.py:91: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  float(human_ce), float(kd_kl_t2), float(combined),


## Cell 5 — 20-step fixed-batch overfit gate


In [5]:
STARTED = time.perf_counter()
reset_student_baseline()

# Teacher distributions are produced on the fly once for each of the four
# fixed images, then detached and cached only for this bounded diagnostic.
FIXED_CACHE = []
all_mass = []
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
for cpu_batch in FIXED_LOADER:
    q, mask, mass = teacher_mapped_distribution(cpu_batch)
    FIXED_CACHE.append({
        "batch": cpu_batch,
        "q": q.detach(),
        "mask": mask.detach(),
    })
    all_mass.extend(mass[mask].detach().cpu().double().tolist())
assert len(FIXED_CACHE) == 4

def fixed_eval():
    STUDENT_ENCODER.eval()
    STUDENT_DECODER.eval()
    rows = []
    with torch.no_grad():
        for item in FIXED_CACHE:
            ce, kd, total = student_losses(
                item["batch"], item["q"], item["mask"]
            )
            rows.append((float(ce), float(kd), float(total)))
    return {
        "human_ce": float(np.mean([row[0] for row in rows])),
        "mapped_kl_t2": float(np.mean([row[1] for row in rows])),
        "combined_loss": float(np.mean([row[2] for row in rows])),
    }

optimizer = torch.optim.AdamW(
    list(STUDENT_ENCODER.parameters())
    + list(STUDENT_DECODER.parameters()),
    lr=1e-3,
    betas=(0.9, 0.999),
    eps=1e-8,
    weight_decay=0.0,
)
initial_eval = fixed_eval()
curve = [{
    "step": 0,
    **initial_eval,
}]
for step in range(1, 21):
    set_student_train()
    optimizer.zero_grad(set_to_none=True)
    for item in FIXED_CACHE:
        ce, kd, total = student_losses(
            item["batch"], item["q"], item["mask"]
        )
        (total / len(FIXED_CACHE)).backward()
    gradient_norm = torch.nn.utils.clip_grad_norm_(
        list(STUDENT_ENCODER.parameters())
        + list(STUDENT_DECODER.parameters()),
        1.0,
    )
    assert torch.isfinite(gradient_norm)
    optimizer.step()
    current_eval = fixed_eval()
    curve.append({
        "step": step,
        **current_eval,
        "gradient_norm_before_clip": float(gradient_norm),
    })
    if step in {1, 5, 10, 15, 20}:
        print(
            f"step={step:02d} | "
            f"combined={current_eval['combined_loss']:.6f} | "
            f"CE={current_eval['human_ce']:.6f} | "
            f"KL*T2={current_eval['mapped_kl_t2']:.6f}"
        )

final_eval = curve[-1]
relative_decrease = (
    initial_eval["combined_loss"] - final_eval["combined_loss"]
) / initial_eval["combined_loss"]
overfit_gate = (
    relative_decrease
    >= float(GATE["overfit_loss_relative_decrease_min"])
)
fixed_mapped_mean = float(np.mean(all_mass))
fixed_mapped_min = float(np.min(all_mass))
fixed_mass_gate = (
    fixed_mapped_mean >= float(GATE["mapped_mass_mean_min"])
    and fixed_mapped_min >= float(GATE["mapped_mass_minimum_min"])
)
overfit_finite_gate = all(
    math.isfinite(row["human_ce"])
    and math.isfinite(row["mapped_kl_t2"])
    and math.isfinite(row["combined_loss"])
    for row in curve
)
overfit_peak_memory_mib = (
    torch.cuda.max_memory_allocated() / (1024 ** 2)
)
overfit_memory_gate = (
    overfit_peak_memory_mib
    <= float(GATE["peak_gpu_memory_mib_max"])
)

curve_path = RESULT / "xlkd02_fixed_batch_overfit_curve.csv"
with curve_path.open("w", encoding="utf-8", newline="") as handle:
    fieldnames = sorted(set().union(*(row.keys() for row in curve)))
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(curve)
OVERFIT_AUDIT = {
    "schema_version": "xlkd02_fixed_batch_overfit_v1",
    "created_at_utc": now(),
    "status": "passed",
    "steps": 20,
    "fixed_images": 4,
    "physical_batch_size": 1,
    "optimizer": "AdamW diagnostic only",
    "learning_rate": 1e-3,
    "weight_decay": 0.0,
    "initial": initial_eval,
    "final": {
        key: value for key, value in final_eval.items()
        if key != "gradient_norm_before_clip"
    },
    "combined_loss_relative_decrease": relative_decrease,
    "mapped_mass_mean": fixed_mapped_mean,
    "mapped_mass_minimum": fixed_mapped_min,
    "peak_gpu_memory_mib": overfit_peak_memory_mib,
    "gates": {
        "finite": overfit_finite_gate,
        "mapped_mass": fixed_mass_gate,
        "relative_decrease_at_least_20_percent": overfit_gate,
        "peak_gpu_memory": overfit_memory_gate,
    },
    "diagnostic_weights_discarded": True,
    "full_training_started": False,
    "validation_read": False,
    "test_set_status": "sealed",
    "test_set_used": False,
}
write_json(RESULT / "xlkd02_fixed_batch_overfit_summary.json", OVERFIT_AUDIT)
write_json(AUDIT / "xlkd02_cell05_overfit.json", {
    **OVERFIT_AUDIT,
    "elapsed_seconds": time.perf_counter() - STARTED,
})
del optimizer
reset_student_baseline()
print("=" * 78)
print("XLKD-02 CELL 5 — FIXED-BATCH OVERFIT")
print("=" * 78)
print("Initial combined loss     :", round(initial_eval["combined_loss"], 6))
print("Final combined loss       :", round(final_eval["combined_loss"], 6))
print("Relative decrease         :", f"{relative_decrease:.2%}")
print("Mapped mass mean / min    :", fixed_mapped_mean, "/", fixed_mapped_min)
print("Peak GPU memory           :", round(overfit_peak_memory_mib, 2), "MiB")
print("Finite / mass / overfit   :", overfit_finite_gate, fixed_mass_gate, overfit_gate)
print("Memory gate               :", overfit_memory_gate)
print("Diagnostic weights        : DISCARDED")
print("Progress                  : 5 / 6 code cells")


step=01 | combined=8.330968 | CE=1.730590 | KL*T2=14.931345
step=05 | combined=5.561234 | CE=0.105738 | KL*T2=11.016730
step=10 | combined=4.000435 | CE=0.080656 | KL*T2=7.920215
step=15 | combined=2.957542 | CE=0.055825 | KL*T2=5.859259
step=20 | combined=2.203443 | CE=0.029687 | KL*T2=4.377199
XLKD-02 CELL 5 — FIXED-BATCH OVERFIT
Initial combined loss     : 8.821373
Final combined loss       : 2.203443
Relative decrease         : 75.02%
Mapped mass mean / min    : 0.9965543732047081 / 0.9942793846130371
Peak GPU memory           : 1223.06 MiB
Finite / mass / overfit   : True True True
Memory gate               : True
Diagnostic weights        : DISCARDED
Progress                  : 5 / 6 code cells


## Cell 6 — Final gate, manifest và XLKD-03 authorization


In [6]:
STARTED = time.perf_counter()
overall_gate = all([
    finite_loss_gate,
    mapped_mass_gate,
    gradient_ratio_gate,
    teacher_gradient_gate,
    memory_gate,
    overfit_finite_gate,
    fixed_mass_gate,
    overfit_gate,
    overfit_memory_gate,
])

DECISION_PATH = RESULT / "xlkd02_decision.json"
write_json(DECISION_PATH, {
    "schema_version": "xlkd02_decision_v1",
    "created_at_utc": now(),
    "status": "passed" if overall_gate else "stopped",
    "overall_gate": overall_gate,
    "gates": {
        "finite_loss_and_gradients": finite_loss_gate,
        "mapped_mass_single_batch": mapped_mass_gate,
        "weighted_gradient_ratio": gradient_ratio_gate,
        "teacher_no_gradient": teacher_gradient_gate,
        "loss_backward_peak_memory": memory_gate,
        "fixed_batch_finite": overfit_finite_gate,
        "fixed_batch_mapped_mass": fixed_mass_gate,
        "fixed_batch_overfit": overfit_gate,
        "fixed_batch_peak_memory": overfit_memory_gate,
    },
    "observed": {
        "gradient_ratio": gradient_ratio,
        "combined_loss_relative_decrease": relative_decrease,
        "peak_gpu_memory_mib": max(
            peak_memory_mib, overfit_peak_memory_mib
        ),
    },
    "decision": (
        "authorize_xlkd03_training_readiness"
        if overall_gate else
        "stop_dense_mapped_logit_kd_before_full_training"
    ),
    "full_training_authorized": False,
    "validation_selection_authorized": False,
    "test_set_status": "sealed",
    "test_set_used": False,
})

write_json(CONFIG / "xlkd02_executed_loss_contract.json", {
    "schema_version": "xlkd02_executed_loss_contract_v1",
    "created_at_utc": now(),
    "status": "locked",
    "teacher": "ExpansionNet v2 frozen",
    "student": "S1 MobileNetV3-Large initialized from seed-42 baseline best",
    "method": "Dense On-the-Fly Cross-Vocabulary Mapped Logit KD",
    "formula": (
        "0.5*CE_human + "
        "0.5*T^2*KL(q_teacher_mapped_renormalized || p_student)"
    ),
    "temperature": 4.0,
    "teacher_conditioning": "native pseudo-caption IDs",
    "student_kd_conditioning": "aligned Phase-1 pseudo-caption IDs",
    "mask_policy": {
        "human_ce": "human target labels",
        "mapped_kl": "aligned pseudo-caption target labels",
    },
    "teacher_probability": (
        "dense mapped distribution generated on the fly; "
        "41 unmapped dimensions discarded then renormalized"
    ),
    "full_training_authorized": False,
    "validation_selection_authorized": False,
    "test_set_status": "sealed",
    "test_set_used": False,
})

if overall_gate:
    NEXT_PATH = OUT / (
        "XLKD03_DENSE_MAPPED_LOGIT_KD_TRAINING_READINESS_AUTHORIZED.json"
    )
    write_json(NEXT_PATH, {
        "schema_version": "xlkd03_dense_mapped_training_readiness_authorized_v1",
        "created_at_utc": now(),
        "status": "authorized",
        "required_next": (
            "XLKD-03 production data/model/optimizer/checkpoint/readiness "
            "contract; discarded pilot only"
        ),
        "method": "Dense On-the-Fly Cross-Vocabulary Mapped Logit KD",
        "student_initialization": "S1 seed-42 baseline best checkpoint",
        "temperature": 4.0,
        "alpha_human_ce": 0.5,
        "alpha_kd": 0.5,
        "physical_batch_size": 1,
        "full_training_authorized": False,
        "validation_selection_authorized": False,
        "test_set_status": "sealed",
        "test_set_used": False,
    })
else:
    NEXT_PATH = OUT / "XLKD_DENSE_MAPPED_LOGIT_KD_BRANCH_STOPPED.json"
    write_json(NEXT_PATH, {
        "schema_version": "xlkd_dense_mapped_branch_stopped_v1",
        "created_at_utc": now(),
        "status": "stopped",
        "reason": "At least one preregistered XLKD-02 gate failed.",
        "failed_gates": [
            key for key, value in read_json(DECISION_PATH)["gates"].items()
            if not value
        ],
        "allowed_next_action": (
            "Report the controlled negative result; do not start full training."
        ),
        "full_training_authorized": False,
        "test_set_status": "sealed",
        "test_set_used": False,
    })

tracked = (
    sorted(AUDIT.glob("*.json"))
    + sorted(CONFIG.glob("*.json"))
    + sorted(RESULT.glob("*"))
)
manifest_rows = [{
    "relative_path": path.relative_to(OUT).as_posix(),
    "size_bytes": path.stat().st_size,
    "sha256": sha256(path),
} for path in tracked]
manifest_path = MANIFEST / "xlkd02_manifest.csv"
with manifest_path.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(
        handle, fieldnames=["relative_path", "size_bytes", "sha256"]
    )
    writer.writeheader()
    writer.writerows(manifest_rows)
sums_path = MANIFEST / "SHA256SUMS"
sums_path.write_text(
    "".join(
        f'{row["sha256"]}  {row["relative_path"]}\n'
        for row in manifest_rows
    ),
    encoding="utf-8",
)

MARKER_PATH = OUT / "XLKD02_DENSE_MAPPED_LOSS_PREFLIGHT_COMPLETE.json"
write_json(MARKER_PATH, {
    "schema_version": "xlkd02_dense_mapped_loss_preflight_complete_v1",
    "created_at_utc": now(),
    "status": "complete",
    "teacher": "ExpansionNet v2",
    "student": "S1 MobileNetV3-Large",
    "method": "Dense On-the-Fly Cross-Vocabulary Mapped Logit KD",
    "overall_gate": overall_gate,
    "gradient_ratio": gradient_ratio,
    "overfit_loss_relative_decrease": relative_decrease,
    "peak_gpu_memory_mib": max(
        peak_memory_mib, overfit_peak_memory_mib
    ),
    "decision": (
        "authorize_xlkd03_training_readiness"
        if overall_gate else
        "stop_dense_mapped_logit_kd_before_full_training"
    ),
    "next_marker": {
        "relative_path": NEXT_PATH.relative_to(OUT).as_posix(),
        "sha256": sha256(NEXT_PATH),
    },
    "full_training_authorized": False,
    "manifest": {
        "csv": {
            "relative_path": manifest_path.relative_to(OUT).as_posix(),
            "sha256": sha256(manifest_path),
            "rows": len(manifest_rows),
        },
        "sha256sums": {
            "relative_path": sums_path.relative_to(OUT).as_posix(),
            "sha256": sha256(sums_path),
        },
    },
    "diagnostic_weights_discarded": True,
    "validation_read": False,
    "test_set_status": "sealed",
    "test_set_used": False,
})
print("=" * 78)
print("XLKD-02 FINAL STATUS")
print("=" * 78)
print("Status                    : COMPLETE")
print("Overall gate              :", "PASSED" if overall_gate else "FAILED")
print("Gradient ratio            :", round(gradient_ratio, 6))
print("20-step loss decrease     :", f"{relative_decrease:.2%}")
print("Peak GPU memory           :", round(max(peak_memory_mib, overfit_peak_memory_mib), 2), "MiB")
print("Decision                  :", read_json(DECISION_PATH)["decision"].upper())
print("Full training             : NOT AUTHORIZED")
print("Validation / Test         : NOT READ / SEALED")
print("Manifest rows             :", len(manifest_rows))
print("Progress                  : 6 / 6 code cells")


XLKD-02 FINAL STATUS
Status                    : COMPLETE
Overall gate              : PASSED
Gradient ratio            : 0.494502
20-step loss decrease     : 75.02%
Peak GPU memory           : 1223.06 MiB
Decision                  : AUTHORIZE_XLKD03_TRAINING_READINESS
Full training             : NOT AUTHORIZED
Validation / Test         : NOT READ / SEALED
Manifest rows             : 11
Progress                  : 6 / 6 code cells
